In [ ]:
#checking the tempelhof #433 dataset
import pandas as pd

# 1. Load the DWD file; skip spaces after semicolons
file_path = r"daily_weather_data/tempelhof_station_daily_weather_04_2025_2026.txt"

df = pd.read_csv(
    file_path,
    sep=";",
    skipinitialspace=True,
    na_values=[-999, "-999", "-999.0"]
)

# 2. Clean column names and remove the DWD end-of-record marker
df.columns = df.columns.str.strip()
df = df.drop(columns=["eor"], errors="ignore")

# 3. Convert the observation date into a proper date
df["MESS_DATUM"] = pd.to_datetime(
    df["MESS_DATUM"].astype("string").str.strip(),
    format="%Y%m%d",
    errors="coerce"
)

# 4. Convert measurement columns to numeric values
for col in df.columns:
    if col not in ["MESS_DATUM", "STATIONS_ID"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 5. Display the cleaned structure
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
print(df.head())

print("\nData types:")
print(df.dtypes)


Dataset shape: (550, 18)

Column names:
['STATIONS_ID', 'MESS_DATUM', 'QN_3', 'FX', 'FM', 'QN_4', 'RSK', 'RSKF', 'SDK', 'SHK_TAG', 'NM', 'VPM', 'PM', 'TMK', 'UPM', 'TXK', 'TNK', 'TGK']

First five rows:
   STATIONS_ID MESS_DATUM  QN_3    FX   FM  QN_4  RSK  RSKF  SDK  SHK_TAG  \
0          433 2025-04-09    10  11.3  3.7     9  0.0     6  NaN      0.0   
1          433 2025-04-10    10  11.3  5.6     9  0.0     0  NaN      0.0   
2          433 2025-04-11    10  15.8  5.8     9  0.0     0  NaN      0.0   
3          433 2025-04-12    10   6.9  2.3     9  0.0     0  NaN      0.0   
4          433 2025-04-13    10  12.0  3.5     9  0.0     6  NaN      0.0   

    NM   VPM      PM   TMK   UPM   TXK  TNK  TGK  
0  7.3   6.8  1016.0   9.4  58.0  15.6  3.4 -0.4  
1  6.5   8.0  1016.6   9.6  68.0  14.4  5.2  3.8  
2  7.8   8.7  1009.8   9.9  71.0  15.3  4.6  0.0  
3  6.3   9.8  1008.1  11.6  73.0  17.9  4.9  1.2  
4  5.3  10.8  1000.7  15.6  63.0  22.7  7.8  3.7  

Data types:
STATIONS_ID    

In [6]:
# 1. Summarise the date range
print("First date:", df["MESS_DATUM"].min())
print("Last date:", df["MESS_DATUM"].max())
print("Number of rows:", len(df))
print("Unique dates:", df["MESS_DATUM"].nunique())

# 2. Check for duplicate dates
duplicate_dates = df.loc[
    df["MESS_DATUM"].duplicated(keep=False),
    "MESS_DATUM"
].sort_values()

print("\nNumber of rows with duplicate dates:", len(duplicate_dates))
print(duplicate_dates.head(10))

# 3. Find missing dates between the first and last observation
valid_dates = df["MESS_DATUM"].dropna()
expected_dates = pd.date_range(
    start=valid_dates.min(),
    end=valid_dates.max(),
    freq="D"
)

observed_dates = pd.DatetimeIndex(valid_dates.unique())
missing_dates = expected_dates.difference(observed_dates)

print("\nExpected daily records:", len(expected_dates))
print("Missing dates:", len(missing_dates))
print("First missing dates:", missing_dates[:15].tolist())


First date: 2025-04-09 00:00:00
Last date: 2026-10-10 00:00:00
Number of rows: 550
Unique dates: 550

Number of rows with duplicate dates: 0
Series([], Name: MESS_DATUM, dtype: datetime64[us])

Expected daily records: 550
Missing dates: 0
First missing dates: []


In [7]:
#Calculate missing values and completeness for each column
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "non_missing_count": df.notna().sum()
})

#Show the least complete columns first
missing_summary = missing_summary.sort_values(
    "missing_percent",
    ascending=False
)

print(missing_summary)

#Check which station IDs are present
print("\nStation IDs:")
print(df["STATIONS_ID"].value_counts(dropna=False))


             missing_count  missing_percent  non_missing_count
SDK                    550           100.00                  0
NM                       8             1.45                542
PM                       5             0.91                545
VPM                      3             0.55                547
UPM                      3             0.55                547
FM                       3             0.55                547
SHK_TAG                  2             0.36                548
FX                       2             0.36                548
TNK                      1             0.18                549
TXK                      1             0.18                549
TGK                      1             0.18                549
RSK                      1             0.18                549
MESS_DATUM               0             0.00                550
STATIONS_ID              0             0.00                550
RSKF                     0             0.00            

In [ ]:
# Summarise the key weather variables
key_variables = ["TMK", "TXK", "TNK", "RSK", "UPM"]

print(df[key_variables].describe().round(2))

# Check whether daily mean temperature falls between min and max
temp_complete = df[["TMK", "TXK", "TNK"]].notna().all(axis=1)

mean_outside_range = df.loc[
    temp_complete
    & (
        (df["TMK"] < df["TNK"] - 0.5)
        | (df["TMK"] > df["TXK"] + 0.5)
    ),
    ["MESS_DATUM", "TNK", "TMK", "TXK"]
]

print("\nMean temperature outside min/max range:")
print(mean_outside_range.head(20))
print("Number of flagged rows:", len(mean_outside_range))

# Flag values that deserve a closer look
# These are screening thresholds, not automatic proof of errors
suspicious = df[
    (df["TXK"] > 45)
    | (df["TNK"] < -30)
    | (df["RSK"] < 0)
    | (df["UPM"] < 0)
    | (df["UPM"] > 100)
]

print("\nRows with potentially suspicious values:")
print(suspicious[["MESS_DATUM"] + key_variables].head(20))
print("Number of flagged rows:", len(suspicious))


          TMK     TXK     TNK     RSK     UPM
count  550.00  549.00  549.00  549.00  547.00
mean    12.94   17.79    7.87    1.24   70.35
std      7.68    9.03    6.77    3.00   13.10
min     -8.10   -6.60  -11.40    0.00   34.00
25%      8.22   12.60    3.30    0.00   61.00
50%     14.35   19.30    9.10    0.00   70.00
75%     18.60   24.10   13.40    1.00   80.00
max     31.70   39.90   23.70   33.60   98.00

Mean temperature outside min/max range:
Empty DataFrame
Columns: [MESS_DATUM, TNK, TMK, TXK]
Index: []
Number of flagged rows: 0

Rows with potentially suspicious values:
Empty DataFrame
Columns: [MESS_DATUM, TMK, TXK, TNK, RSK, UPM]
Index: []
Number of flagged rows: 0


In [11]:
# 1. Select the summer period for both years
summer_start = "2025-06-01"
summer_end = "2025-09-30"

summer_2025 = df[
    df["MESS_DATUM"].between(summer_start, summer_end)
]

summer_2026 = df[
    df["MESS_DATUM"].between("2026-06-01", "2026-09-30")
]

# 2. Compare the number of records and temperature coverage
for label, subset in [
    ("Summer 2025", summer_2025),
    ("Summer 2026", summer_2026)
]:
    print(f"\n{label}")
    print("Rows:", len(subset))
    print("Days with maximum temperature:", subset["TXK"].notna().sum())
    print("Days with minimum temperature:", subset["TNK"].notna().sum())
    print("Days with mean temperature:", subset["TMK"].notna().sum())
    print("Days with precipitation:", subset["RSK"].notna().sum())



Summer 2025
Rows: 122
Days with maximum temperature: 122
Days with minimum temperature: 122
Days with mean temperature: 122
Days with precipitation: 122

Summer 2026
Rows: 122
Days with maximum temperature: 122
Days with minimum temperature: 122
Days with mean temperature: 122
Days with precipitation: 121


In [12]:
# Show the dates and measurements around the highest recorded temperature
hottest_day = df.loc[df["TXK"].idxmax()]

print("Hottest day in the dataset:")
print(hottest_day[["STATIONS_ID", "MESS_DATUM", "TMK", "TXK", "TNK", "RSK"]])

# Show the station ID and the date range
print("\nStation IDs:", df["STATIONS_ID"].unique())
print("Date range:", df["MESS_DATUM"].min(), "to", df["MESS_DATUM"].max())

Hottest day in the dataset:
STATIONS_ID                    433
MESS_DATUM     2026-06-27 00:00:00
TMK                           31.4
TXK                           39.9
TNK                           22.0
RSK                            0.1
Name: 444, dtype: object

Station IDs: [433]
Date range: 2025-04-09 00:00:00 to 2026-10-10 00:00:00


In [ ]:
# Count the quality flags in each column
for col in ["QN_3", "QN_4"]:
    if col in df.columns:
        print(f"\n{col} value counts:")
        print(df[col].value_counts(dropna=False).sort_index())

# Check quality flags alongside the main temperature variables
print("\nQuality flags and temperature measurements:")
print(
    df[["MESS_DATUM", "QN_3", "QN_4", "TMK", "TXK", "TNK"]]
    .head(10)
)



QN_3 value counts:
QN_3
1      73
3      90
10    387
Name: count, dtype: int64

QN_4 value counts:
QN_4
1     73
3    151
9    326
Name: count, dtype: int64

Quality flags and temperature measurements:
  MESS_DATUM  QN_3  QN_4   TMK   TXK   TNK
0 2025-04-09    10     9   9.4  15.6   3.4
1 2025-04-10    10     9   9.6  14.4   5.2
2 2025-04-11    10     9   9.9  15.3   4.6
3 2025-04-12    10     9  11.6  17.9   4.9
4 2025-04-13    10     9  15.6  22.7   7.8
5 2025-04-14    10     9  15.7  21.5   7.9
6 2025-04-15    10     9  16.9  22.0  11.4
7 2025-04-16    10     9  19.1  25.9  12.6
8 2025-04-17    10     9  18.9  26.7  13.4
9 2025-04-18    10     9  11.8  13.8   8.7


In [ ]:
# Compare temperature completeness by quality-flag value
for flag_col in ["QN_3", "QN_4"]:
    print(f"\nTemperature completeness by {flag_col}")

    # Count available and missing temperature measurements for each flag
    summary = df.groupby(flag_col)[["TMK", "TXK", "TNK"]].agg(
        ["count", lambda s: s.isna().sum()]
    )

    print(summary)



Temperature completeness by QN_3
       TMK              TXK              TNK           
     count <lambda_0> count <lambda_0> count <lambda_0>
QN_3                                                   
1       73          0    72          1    72          1
3       90          0    90          0    90          0
10     387          0   387          0   387          0

Temperature completeness by QN_4
       TMK              TXK              TNK           
     count <lambda_0> count <lambda_0> count <lambda_0>
QN_4                                                   
1       73          0    72          1    72          1
3      151          0   151          0   151          0
9      326          0   326          0   326          0


In [ ]:
# Find records where maximum or minimum temperature is missing
missing_temp = df.loc[
    df[["TXK", "TNK"]].isna().any(axis=1),
    ["MESS_DATUM", "STATIONS_ID", "QN_3", "QN_4",
     "TMK", "TXK", "TNK", "RSK"]
]

print(missing_temp.to_string(index=False))


MESS_DATUM  STATIONS_ID  QN_3  QN_4  TMK  TXK  TNK  RSK
2026-10-07          433     1     1 15.2  NaN  NaN  0.0


In [16]:
# Define the summer period for each year
summer_2025 = df[
    (df["MESS_DATUM"] >= "2025-06-01") &
    (df["MESS_DATUM"] <= "2025-08-31")
]

summer_2026 = df[
    (df["MESS_DATUM"] >= "2026-06-01") &
    (df["MESS_DATUM"] <= "2026-08-31")
]

# Show the five hottest days by maximum temperature
print("Summer 2025 — five hottest days:")
print(
    summer_2025.nlargest(5, "TXK")[
        ["MESS_DATUM", "TXK", "TNK", "TMK"]
    ].to_string(index=False)
)

print("\nSummer 2026 — five hottest days:")
print(
    summer_2026.nlargest(5, "TXK")[
        ["MESS_DATUM", "TXK", "TNK", "TMK"]
    ].to_string(index=False)
)

Summer 2025 — five hottest days:
MESS_DATUM  TXK  TNK  TMK
2025-07-02 37.6 17.9 29.2
2025-06-22 34.7 14.9 26.5
2025-08-15 34.5 18.0 26.6
2025-08-14 33.6 17.2 25.7
2025-06-26 33.2 13.9 22.6

Summer 2026 — five hottest days:
MESS_DATUM  TXK  TNK  TMK
2026-06-27 39.9 22.0 31.4
2026-06-28 39.6 23.7 31.7
2026-07-30 38.0 17.8 28.4
2026-08-15 36.7 16.7 26.5
2026-06-19 35.3 17.7 26.3


In [17]:
# Summarise temperatures and count hot days for each summer
for year, summer in [(2025, summer_2025), (2026, summer_2026)]:
    print(f"\nSummer {year}")
    print(f"Days in dataset: {len(summer)}")
    print(f"Days with maximum temperature >= 30°C: {(summer['TXK'] >= 30).sum()}")
    print(f"Days with maximum temperature >= 35°C: {(summer['TXK'] >= 35).sum()}")
    print(f"Average daily maximum: {summer['TXK'].mean():.1f}°C")
    print(f"Average daily mean: {summer['TMK'].mean():.1f}°C")
    print(f"Average daily minimum: {summer['TNK'].mean():.1f}°C")


Summer 2025
Days in dataset: 92
Days with maximum temperature >= 30°C: 11
Days with maximum temperature >= 35°C: 1
Average daily maximum: 25.1°C
Average daily mean: 19.6°C
Average daily minimum: 13.9°C

Summer 2026
Days in dataset: 92
Days with maximum temperature >= 30°C: 21
Days with maximum temperature >= 35°C: 6
Average daily maximum: 26.3°C
Average daily mean: 20.4°C
Average daily minimum: 14.3°C


In [18]:
# Calculate the longest consecutive run of days reaching 30°C
for year, summer in [(2025, summer_2025), (2026, summer_2026)]:
    # Sort by date to ensure the sequence is chronological
    summer = summer.sort_values("MESS_DATUM").copy()

    # Mark each day as hot (True) or not hot (False)
    hot = summer["TXK"].ge(30)

    # Start a new group whenever the hot/not-hot status changes
    groups = hot.ne(hot.shift()).cumsum()

    # Count consecutive hot days within each group
    hot_runs = hot.groupby(groups).sum()

    # Keep only groups that contain hot days
    longest_run = int(hot_runs.max()) if hot.any() else 0

    print(f"Summer {year}: longest run of days >= 30°C = {longest_run}")

Summer 2025: longest run of days >= 30°C = 3
Summer 2026: longest run of days >= 30°C = 5


In [24]:
#checking data for Buch #400 weather station
#Load the DWD file; skip spaces after semicolons
file_path = r"daily_weather_data/buch_station_daily_weather_04_2025_2026.txt"

df_400 = pd.read_csv(
    file_path,
    sep=";",
    skipinitialspace=True,
    na_values=[-999, "-999", "-999.0"]
)

#Clean column names and remove the DWD end-of-record marker
df_400.columns = df_400.columns.str.strip()
df_400 = df_400.drop(columns=["eor"], errors="ignore")

#Convert the observation date into a proper date
df_400["MESS_DATUM"] = pd.to_datetime(
    df_400["MESS_DATUM"].astype("string").str.strip(),
    format="%Y%m%d",
    errors="coerce"
)

#Convert measurement columns to numeric values
for col in df_400.columns:
    if col not in ["MESS_DATUM", "STATIONS_ID"]:
        df_400[col] = pd.to_numeric(df_400[col], errors="coerce")

#Display the cleaned structure
print("Dataset shape:", df_400.shape)
print("\nColumn names:")
print(df_400.columns.tolist())

print("\nFirst five rows:")
print(df_400.head())

print("\nData types:")
print(df_400.dtypes)


Dataset shape: (550, 18)

Column names:
['STATIONS_ID', 'MESS_DATUM', 'QN_3', 'FX', 'FM', 'QN_4', 'RSK', 'RSKF', 'SDK', 'SHK_TAG', 'NM', 'VPM', 'PM', 'TMK', 'UPM', 'TXK', 'TNK', 'TGK']

First five rows:
   STATIONS_ID MESS_DATUM  QN_3  FX  FM  QN_4  RSK  RSKF  SDK  SHK_TAG  NM  \
0          400 2025-04-09   NaN NaN NaN    10  0.0     0  NaN      0.0 NaN   
1          400 2025-04-10   NaN NaN NaN    10  0.0     0  NaN      0.0 NaN   
2          400 2025-04-11   NaN NaN NaN    10  0.0     0  NaN      0.0 NaN   
3          400 2025-04-12   NaN NaN NaN    10  0.0     0  NaN      0.0 NaN   
4          400 2025-04-13   NaN NaN NaN    10  0.0     4  NaN      0.0 NaN   

    VPM  PM   TMK   UPM   TXK  TNK  TGK  
0   6.9 NaN   8.8  62.0  14.4  3.2  0.2  
1   7.8 NaN   9.0  69.0  13.9  3.3  0.4  
2   8.7 NaN   9.2  75.0  14.9  5.2  1.6  
3   9.6 NaN  11.3  74.0  18.1  6.0  3.1  
4  10.7 NaN  15.1  64.0  22.7  7.0  2.8  

Data types:
STATIONS_ID             int64
MESS_DATUM     datetime64[us]
QN_

In [27]:
# 1. Summarise the date range - ctrl+shift+L if you want to change a specific variable in the box everywhere
print("First date:", df_400["MESS_DATUM"].min())
print("Last date:", df_400["MESS_DATUM"].max())
print("Number of rows:", len(df_400))
print("Unique dates:", df_400["MESS_DATUM"].nunique())

# 2. Check for duplicate dates
duplicate_dates = df_400.loc[
    df_400["MESS_DATUM"].duplicated(keep=False),
    "MESS_DATUM"
].sort_values()

print("\nNumber of rows with duplicate dates:", len(duplicate_dates))
print(duplicate_dates.head(10))

# 3. Find missing dates between the first and last observation
valid_dates_400 = df_400["MESS_DATUM"].dropna()
expected_dates = pd.date_range(
    start=valid_dates.min(),
    end=valid_dates.max(),
    freq="D"
)

observed_dates_400 = pd.DatetimeIndex(valid_dates.unique())
missing_dates_400 = expected_dates.difference(observed_dates)

print("\nExpected daily records:", len(expected_dates))
print("Missing dates:", len(missing_dates))
print("First missing dates:", missing_dates[:15].tolist())


First date: 2025-04-09 00:00:00
Last date: 2026-10-10 00:00:00
Number of rows: 550
Unique dates: 550

Number of rows with duplicate dates: 0
Series([], Name: MESS_DATUM, dtype: datetime64[us])

Expected daily records: 550
Missing dates: 0
First missing dates: []


In [32]:
#Calculate missing values and completeness for each column
missing_summary_400 = pd.DataFrame({
    "missing_count": df_400.isna().sum(),
    "missing_percent": (df_400.isna().mean() * 100).round(2),
    "non_missing_count": df_400.notna().sum()
})

#Show the least complete columns first
missing_summary_400 = missing_summary_400.sort_values(
    "missing_percent",
    ascending=False
)

print(missing_summary_400)

#Check which station IDs are present
print("\nStation IDs:")
print(df_400["STATIONS_ID"].value_counts(dropna=False))



             missing_count  missing_percent  non_missing_count
FX                     550           100.00                  0
QN_3                   550           100.00                  0
PM                     550           100.00                  0
NM                     550           100.00                  0
FM                     550           100.00                  0
SDK                    550           100.00                  0
SHK_TAG                306            55.64                244
STATIONS_ID              0             0.00                550
MESS_DATUM               0             0.00                550
QN_4                     0             0.00                550
RSK                      0             0.00                550
RSKF                     0             0.00                550
VPM                      0             0.00                550
TMK                      0             0.00                550
UPM                      0             0.00            

In [35]:
# Summarise the key weather variables
key_variables_400 = ["TMK", "TXK", "TNK", "RSK", "UPM"]

print(df_400[key_variables].describe().round(2))

# Check whether daily mean temperature falls between min and max
temp_complete_400 = df_400[["TMK", "TXK", "TNK"]].notna().all(axis=1)

mean_outside_range_400 = df_400.loc[
    temp_complete_400
    & (
        (df_400["TMK"] < df_400["TNK"] - 0.5)
        | (df_400["TMK"] > df_400["TXK"] + 0.5)
    ),
    ["MESS_DATUM", "TNK", "TMK", "TXK"]
]

print("\nMean temperature outside min/max range:")
print(mean_outside_range_400.head(20))
print("Number of flagged rows:", len(mean_outside_range_400))

# Flag values that deserve a closer look
# These are screening thresholds, not automatic proof of errors
suspicious_400 = df_400[
    (df_400["TXK"] > 45)
    | (df_400["TNK"] < -30)
    | (df_400["RSK"] < 0)
    | (df_400["UPM"] < 0)
    | (df_400["UPM"] > 100)
]

print("\nRows with potentially suspicious values:")
print(suspicious_400[["MESS_DATUM"] + key_variables_400].head(20))
print("Number of flagged rows:", len(suspicious_400))


          TMK     TXK     TNK     RSK     UPM
count  550.00  550.00  550.00  550.00  550.00
mean    12.17   17.43    6.89    1.46   72.88
std      7.51    9.02    6.54    3.80   12.75
min     -9.10   -7.30  -12.60    0.00   36.00
25%      7.43   12.22    2.70    0.00   63.84
50%     13.50   19.00    7.90    0.00   73.00
75%     17.70   23.68   12.00    1.00   82.86
max     30.40   39.80   21.80   42.90   98.00

Mean temperature outside min/max range:
Empty DataFrame
Columns: [MESS_DATUM, TNK, TMK, TXK]
Index: []
Number of flagged rows: 0

Rows with potentially suspicious values:
Empty DataFrame
Columns: [MESS_DATUM, TMK, TXK, TNK, RSK, UPM]
Index: []
Number of flagged rows: 0


In [36]:
# 1. Select the summer period for both years
summer_start = "2025-06-01"
summer_end = "2025-09-30"

summer_2025 = df_400[
    df_400["MESS_DATUM"].between(summer_start, summer_end)
]

summer_2026 = df_400[
    df_400["MESS_DATUM"].between("2026-06-01", "2026-09-30")
]

# 2. Compare the number of records and temperature coverage
for label, subset in [
    ("Summer 2025", summer_2025),
    ("Summer 2026", summer_2026)
]:
    print(f"\n{label}")
    print("Rows:", len(subset))
    print("Days with maximum temperature:", subset["TXK"].notna().sum())
    print("Days with minimum temperature:", subset["TNK"].notna().sum())
    print("Days with mean temperature:", subset["TMK"].notna().sum())
    print("Days with precipitation:", subset["RSK"].notna().sum())



Summer 2025
Rows: 122
Days with maximum temperature: 122
Days with minimum temperature: 122
Days with mean temperature: 122
Days with precipitation: 122

Summer 2026
Rows: 122
Days with maximum temperature: 122
Days with minimum temperature: 122
Days with mean temperature: 122
Days with precipitation: 122


In [46]:
# Show the dates and measurements around the highest recorded temperature
hottest_day_400 = df_400.loc[df_400["TXK"].idxmax()]

print("Hottest day in the dataset:")
print(hottest_day_400[["STATIONS_ID", "MESS_DATUM", "TMK", "TXK", "TNK", "RSK"]])

# Show the station ID and the date range
print("\nStation IDs:", df_400["STATIONS_ID"].unique())
print("Date range:", df_400["MESS_DATUM"].min(), "to", df_400["MESS_DATUM"].max())

Hottest day in the dataset:
STATIONS_ID                    400
MESS_DATUM     2026-06-27 00:00:00
TMK                           30.2
TXK                           39.8
TNK                           19.0
RSK                            0.0
Name: 444, dtype: object

Station IDs: [400]
Date range: 2025-04-09 00:00:00 to 2026-10-10 00:00:00


In [38]:
# Count the quality flags in each column
for col in ["QN_3", "QN_4"]:
    if col in df_400.columns:
        print(f"\n{col} value counts:")
        print(df_400[col].value_counts(dropna=False).sort_index())

# Check quality flags alongside the main temperature variables
print("\nQuality flags and temperature measurements:")
print(
    df_400[["MESS_DATUM", "QN_3", "QN_4", "TMK", "TXK", "TNK"]]
    .head(10)
)



QN_3 value counts:
QN_3
NaN    550
Name: count, dtype: int64

QN_4 value counts:
QN_4
1      73
3     151
10    326
Name: count, dtype: int64

Quality flags and temperature measurements:
  MESS_DATUM  QN_3  QN_4   TMK   TXK   TNK
0 2025-04-09   NaN    10   8.8  14.4   3.2
1 2025-04-10   NaN    10   9.0  13.9   3.3
2 2025-04-11   NaN    10   9.2  14.9   5.2
3 2025-04-12   NaN    10  11.3  18.1   6.0
4 2025-04-13   NaN    10  15.1  22.7   7.0
5 2025-04-14   NaN    10  15.0  21.1   8.1
6 2025-04-15   NaN    10  17.0  23.3  11.6
7 2025-04-16   NaN    10  18.6  25.8  10.4
8 2025-04-17   NaN    10  19.4  27.2  13.3
9 2025-04-18   NaN    10  11.8  14.0   8.5


In [39]:
# Compare temperature completeness by quality-flag value
for flag_col in ["QN_3", "QN_4"]:
    print(f"\nTemperature completeness by {flag_col}")

    # Count available and missing temperature measurements for each flag
    summary_400 = df_400.groupby(flag_col)[["TMK", "TXK", "TNK"]].agg(
        ["count", lambda s: s.isna().sum()]
    )

    print(summary_400)



Temperature completeness by QN_3
Empty DataFrame
Columns: [(TMK, count), (TMK, <lambda_0>), (TXK, count), (TXK, <lambda_0>), (TNK, count), (TNK, <lambda_0>)]
Index: []

Temperature completeness by QN_4
       TMK              TXK              TNK           
     count <lambda_0> count <lambda_0> count <lambda_0>
QN_4                                                   
1       73          0    73          0    73          0
3      151          0   151          0   151          0
10     326          0   326          0   326          0


In [42]:
# Find records where maximum or minimum temperature is missing
missing_temp_400 = df_400.loc[
    df_400[["TXK", "TNK"]].isna().any(axis=1),
    ["MESS_DATUM", "STATIONS_ID", "QN_3", "QN_4",
     "TMK", "TXK", "TNK", "RSK"]
]

print(missing_temp_400.to_string(index=False))


Empty DataFrame
Columns: [MESS_DATUM, STATIONS_ID, QN_3, QN_4, TMK, TXK, TNK, RSK]
Index: []


In [43]:
# Define the summer period for each year
summer_2025 = df_400[
    (df_400["MESS_DATUM"] >= "2025-06-01") &
    (df_400["MESS_DATUM"] <= "2025-08-31")
]

summer_2026 = df_400[
    (df_400["MESS_DATUM"] >= "2026-06-01") &
    (df_400["MESS_DATUM"] <= "2026-08-31")
]

# Show the five hottest days by maximum temperature
print("Summer 2025 — five hottest days:")
print(
    summer_2025.nlargest(5, "TXK")[
        ["MESS_DATUM", "TXK", "TNK", "TMK"]
    ].to_string(index=False)
)

print("\nSummer 2026 — five hottest days:")
print(
    summer_2026.nlargest(5, "TXK")[
        ["MESS_DATUM", "TXK", "TNK", "TMK"]
    ].to_string(index=False)
)

Summer 2025 — five hottest days:
MESS_DATUM  TXK  TNK  TMK
2025-07-02 37.6 16.2 27.9
2025-08-15 34.1 17.5 26.1
2025-06-22 33.9 12.6 24.7
2025-08-14 33.4 14.9 24.4
2025-07-01 32.9 12.2 23.2

Summer 2026 — five hottest days:
MESS_DATUM  TXK  TNK  TMK
2026-06-27 39.8 19.0 30.2
2026-06-28 39.2 21.8 30.4
2026-07-30 37.9 17.6 27.8
2026-08-15 35.7 15.3 25.4
2026-06-25 34.7 16.4 26.4


In [44]:
# Summarise temperatures and count hot days for each summer
for year, summer in [(2025, summer_2025), (2026, summer_2026)]:
    print(f"\nSummer {year}")
    print(f"Days in dataset: {len(summer)}")
    print(f"Days with maximum temperature >= 30°C: {(summer['TXK'] >= 30).sum()}")
    print(f"Days with maximum temperature >= 35°C: {(summer['TXK'] >= 35).sum()}")
    print(f"Average daily maximum: {summer['TXK'].mean():.1f}°C")
    print(f"Average daily mean: {summer['TMK'].mean():.1f}°C")
    print(f"Average daily minimum: {summer['TNK'].mean():.1f}°C")


Summer 2025
Days in dataset: 92
Days with maximum temperature >= 30°C: 9
Days with maximum temperature >= 35°C: 1
Average daily maximum: 24.6°C
Average daily mean: 18.6°C
Average daily minimum: 12.5°C

Summer 2026
Days in dataset: 92
Days with maximum temperature >= 30°C: 20
Days with maximum temperature >= 35°C: 4
Average daily maximum: 25.9°C
Average daily mean: 19.5°C
Average daily minimum: 13.1°C


In [45]:
# Calculate the longest consecutive run of days reaching 30°C
for year, summer in [(2025, summer_2025), (2026, summer_2026)]:
    # Sort by date to ensure the sequence is chronological
    summer = summer.sort_values("MESS_DATUM").copy()

    # Mark each day as hot (True) or not hot (False)
    hot = summer["TXK"].ge(30)

    # Start a new group whenever the hot/not-hot status changes
    groups = hot.ne(hot.shift()).cumsum()

    # Count consecutive hot days within each group
    hot_runs = hot.groupby(groups).sum()

    # Keep only groups that contain hot days
    longest_run = int(hot_runs.max()) if hot.any() else 0

    print(f"Summer {year}: longest run of days >= 30°C = {longest_run}")

Summer 2025: longest run of days >= 30°C = 3
Summer 2026: longest run of days >= 30°C = 5


In [ ]:
#function to load each dataset

def load_daily_weather(file_path):
    # Load the DWD semicolon-separated file and treat -999 as missing
    df = pd.read_csv(
        file_path,
        sep=";",
        skipinitialspace=True,
        na_values=[-999, "-999", "-999.0"]
    )

    # Clean column names and remove the end-of-record marker
    df.columns = df.columns.str.strip()
    df = df.drop(columns=["eor"], errors="ignore")

    # Convert the date column to datetime format
    df["MESS_DATUM"] = pd.to_datetime(
        df["MESS_DATUM"].astype("string").str.strip(),
        format="%Y%m%d",
        errors="coerce"
    )

    # Convert all other columns to numeric values
    for col in df.columns:
        if col != "MESS_DATUM":
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

[403]


In [ ]:
#trying a function to validate the other datasets

def validate_daily_weather(df, station_name="Unknown station"):
    # Work on a copy so the original dataset is not modified
    data = df.copy()

    # Clean column names and convert measurement columns to numeric
    data.columns = data.columns.str.strip()
    data["MESS_DATUM"] = pd.to_datetime(data["MESS_DATUM"], errors="coerce")

    for col in data.columns:
        if col not in ["MESS_DATUM"]:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    print("=" * 60)
    print(f"DAILY WEATHER VALIDATION: {station_name}")
    print("=" * 60)

    # 1. Check station IDs, date range and dataset size
    print("\n1. DATA OVERVIEW")
    print("Station IDs:", data["STATIONS_ID"].dropna().unique())
    print("First date:", data["MESS_DATUM"].min())
    print("Last date:", data["MESS_DATUM"].max())
    print("Rows:", len(data))
    print("Unique dates:", data["MESS_DATUM"].nunique())

    # 2. Check duplicate dates and missing dates
    print("\n2. DATE COVERAGE")
    print("Duplicate date rows:", data["MESS_DATUM"].duplicated().sum())

    valid_dates = data["MESS_DATUM"].dropna()
    if not valid_dates.empty:
        expected_dates = pd.date_range(
            valid_dates.min(), valid_dates.max(), freq="D"
        )
        missing_dates = expected_dates.difference(valid_dates)
        print("Missing dates:", len(missing_dates))
        if len(missing_dates) > 0:
            print("First missing dates:", list(missing_dates[:10]))

    # 3. Report missing values by column
    print("\n3. MISSING VALUES")
    missing = pd.DataFrame({
        "missing_count": data.isna().sum(),
        "missing_percent": (data.isna().mean() * 100).round(2)
    })
    print(missing[missing["missing_count"] > 0].sort_values(
        "missing_percent", ascending=False
    ).to_string())

    # 4. Summarise key weather measurements
    print("\n4. TEMPERATURE AND WEATHER SUMMARY")
    core_columns = ["TMK", "TXK", "TNK", "RSK", "UPM"]
    available_columns = [c for c in core_columns if c in data.columns]
    print(data[available_columns].describe().round(2).to_string())

    # 5. Check whether mean temperature falls between min and max
    if all(c in data.columns for c in ["TNK", "TMK", "TXK"]):
        # Allow a small tolerance for rounding differences
        invalid_temp = data[
            (data["TMK"] < data["TNK"] - 0.5) |
            (data["TMK"] > data["TXK"] + 0.5) |
            (data["TNK"] > data["TXK"] + 0.5)
        ]
        print("\n5. TEMPERATURE CONSISTENCY")
        print("Potentially inconsistent rows:", len(invalid_temp))
        if not invalid_temp.empty:
            print(invalid_temp[
                ["MESS_DATUM", "TNK", "TMK", "TXK"]
            ].head(10).to_string(index=False))

    # 6. Screen for simple suspicious values
    print("\n6. BASIC PLAUSIBILITY SCREEN")
    if "TXK" in data.columns:
        print("Maximum temperature above 45°C:",
              (data["TXK"] > 45).sum())
    if "TNK" in data.columns:
        print("Minimum temperature below -30°C:",
              (data["TNK"] < -30).sum())
    if "RSK" in data.columns:
        print("Negative precipitation:", (data["RSK"] < 0).sum())
    if "UPM" in data.columns:
        print("Humidity outside 0–100%:",
              ((data["UPM"] < 0) | (data["UPM"] > 100)).sum())

    # 7. Compare June, July and August for 2025 and 2026
    print("\n7. SUMMER COMPARISON")
    data["MESS_DATUM"] = pd.to_datetime(data["MESS_DATUM"])
    for year in [2025, 2026]:
        summer = data[
            (data["MESS_DATUM"] >= f"{year}-06-01") &
            (data["MESS_DATUM"] <= f"{year}-08-31")
        ].sort_values("MESS_DATUM")

        print(f"\nSummer {year}")
        print("Rows:", len(summer))

        if "TXK" not in summer.columns:
            continue

        # Count hot days and calculate average temperatures
        print("Days with TXK:", summer["TXK"].notna().sum())
        print("Days >= 30°C:", (summer["TXK"] >= 30).sum())
        print("Days >= 35°C:", (summer["TXK"] >= 35).sum())

        for col, label in [
            ("TXK", "Average daily maximum"),
            ("TMK", "Average daily mean"),
            ("TNK", "Average daily minimum")
        ]:
            if col in summer.columns:
                print(f"{label}: {summer[col].mean():.1f}°C")

        # Find the five hottest days
        print("\nFive hottest days:")
        show_columns = [
            c for c in ["MESS_DATUM", "TXK", "TNK", "TMK"]
            if c in summer.columns
        ]
        print(summer.nlargest(5, "TXK")[show_columns].to_string(
            index=False
        ))

        # Calculate the longest run of consecutive hot calendar days
        daily = summer.dropna(subset=["MESS_DATUM"]).copy()
        daily["date"] = daily["MESS_DATUM"].dt.normalize()
        daily = daily.drop_duplicates("date").set_index("date")
        calendar = pd.date_range(
            f"{year}-06-01", f"{year}-08-31", freq="D"
        )
        daily = daily.reindex(calendar)

        # Missing dates break a run rather than being treated as cool days
        hot = daily["TXK"].ge(30).fillna(False)
        observed = daily["TXK"].notna()
        run_lengths = []
        current_run = 0

        for is_observed, is_hot in zip(observed, hot):
            if is_observed and is_hot:
                current_run += 1
            else:
                run_lengths.append(current_run)
                current_run = 0
        run_lengths.append(current_run)

        print("Longest consecutive run >= 30°C:",
              max(run_lengths, default=0), "days")

        # Show the hottest individual day in this summer
        if summer["TXK"].notna().any():
            hottest = summer.loc[summer["TXK"].idxmax()]
            print("\nHottest day:")
            print(hottest[
                [c for c in [
                    "STATIONS_ID", "MESS_DATUM", "TMK",
                    "TXK", "TNK", "RSK"
                ] if c in summer.columns]
            ].to_string())

    # 8. Summarise quality-flag values without filtering records
    print("\n8. QUALITY FLAGS")
    for col in ["QN_3", "QN_4"]:
        if col in data.columns:
            print(f"\n{col}:")
            print(data[col].value_counts(dropna=False).to_string())

    print("\nValidation finished.")


In [ ]:
#using the first function to load the Dahlem dataset and saving it into a variable:
df_403 = load_daily_weather("daily_weather_data/dahlem_station_daily_weather_04_2025_2026.txt")

#using the second function to validate the new dataset:
validate_daily_weather(df_403, station_name="Dahlem 403 Station")

DAILY WEATHER VALIDATION: Dahlem 403 Station

1. DATA OVERVIEW
Station IDs: [403]
First date: 2025-04-09 00:00:00
Last date: 2026-10-10 00:00:00
Rows: 550
Unique dates: 550

2. DATE COVERAGE
Duplicate date rows: 0
Missing dates: 0

3. MISSING VALUES
      missing_count  missing_percent
QN_3            550           100.00
FX              550           100.00
FM              550           100.00
NM                3             0.55

4. TEMPERATURE AND WEATHER SUMMARY
          TMK     TXK     TNK     RSK     UPM
count  550.00  550.00  550.00  550.00  550.00
mean    12.40   17.44    7.49    1.34   75.32
std      7.43    8.81    6.54    3.60   13.45
min     -7.80   -6.20  -12.00    0.00   39.00
25%      7.90   12.35    3.00    0.00   65.41
50%     13.85   19.10    8.70    0.00   75.00
75%     17.90   23.58   12.60    0.80   86.00
max     29.70   39.00   21.20   44.60  100.00

5. TEMPERATURE CONSISTENCY
Potentially inconsistent rows: 0

6. BASIC PLAUSIBILITY SCREEN
Maximum temperature abov

In [57]:
df_420 = load_daily_weather("daily_weather_data/marzahn_station_daily_weather_04_2025_2026.txt")

validate_daily_weather(df_420, station_name="Marzahn 420 Station")

DAILY WEATHER VALIDATION: Marzahn 420 Station

1. DATA OVERVIEW
Station IDs: [420]
First date: 2025-04-09 00:00:00
Last date: 2026-10-10 00:00:00
Rows: 550
Unique dates: 550

2. DATE COVERAGE
Duplicate date rows: 0
Missing dates: 0

3. MISSING VALUES
         missing_count  missing_percent
QN_3               550           100.00
FX                 550           100.00
FM                 550           100.00
SHK_TAG            550           100.00
SDK                550           100.00
PM                 550           100.00
NM                 550           100.00
VPM                  9             1.64
UPM                  9             1.64
RSK                  7             1.27
RSKF                 7             1.27
TMK                  7             1.27
TXK                  7             1.27
TNK                  7             1.27
TGK                  7             1.27

4. TEMPERATURE AND WEATHER SUMMARY
          TMK     TXK     TNK     RSK     UPM
count  543.00  543.00  543.

In [58]:
df_427 = load_daily_weather("daily_weather_data/berlin_brandenburg_station_daily_weather_04_2025_2026.txt")

validate_daily_weather(df_427, station_name="Berlin Brandenburg 427 Station")

DAILY WEATHER VALIDATION: Berlin Brandenburg 427 Station

1. DATA OVERVIEW
Station IDs: [427]
First date: 2025-04-09 00:00:00
Last date: 2026-10-10 00:00:00
Rows: 550
Unique dates: 550

2. DATE COVERAGE
Duplicate date rows: 0
Missing dates: 0

3. MISSING VALUES
         missing_count  missing_percent
SDK                550           100.00
FM                  11             2.00
PM                  10             1.82
FX                   9             1.64
NM                   9             1.64
VPM                  7             1.27
UPM                  7             1.27
SHK_TAG              3             0.55

4. TEMPERATURE AND WEATHER SUMMARY
          TMK     TXK     TNK     RSK     UPM
count  550.00  550.00  550.00  550.00  543.00
mean    12.50   17.61    7.03    1.26   72.38
std      7.73    9.00    6.83    3.25   13.54
min     -8.00   -6.40  -14.10    0.00   38.00
25%      7.60   12.22    2.22    0.00   62.38
50%     13.90   19.30    8.15    0.00   72.21
75%     18.27   23.7